# report_data

> Build the data layer for the single-page evaluation report (#79): one aggregates-only
> JSON assembled from a pipeline output directory, plus the renderer that injects it into
> the self-contained HTML template. Replaces the render-time Quarto/papermill/SHAP path.


In [ ]:
#| default_exp report_data

In [ ]:
#| export
from __future__ import annotations

import json
import re
from collections import Counter, defaultdict
from typing import Any
from pathlib import Path

import numpy as np
import pandas as pd
import structlog

from kreview.pipeline_diagram import (
    CLUSTERS,
    mini_store,
    node_names,
    node_process_map,
    pipeline_svg,
)

log = structlog.get_logger()

# Subgroup metrics are emitted only when the group has at least this many samples —
# below it the AUC is noise, and tiny groups edge toward re-identifiability.
# Declared a priori in ANALYSIS_PLAN.md so the report can show the pre-registered
# primary next to the run's argmax: best-of-156 selection inflates the winner by
# ~0.01–0.02 AUC, so the argmax evaluator/model is exploratory, not confirmatory.
PRIMARY_EVALUATOR = "FSCGenomewide"
PRIMARY_MODEL = "tabicl_ft"

# Detection-vs-burden bins (plasma max VAF as a fraction). Chosen to bracket the
# assay's transition zone: flat below 1%, steep 1-10%, saturated above.
LOD_BINS = (
    (0.0, 0.001, "<0.1%"),
    (0.001, 0.005, "0.1-0.5%"),
    (0.005, 0.01, "0.5-1%"),
    (0.01, 0.02, "1-2%"),
    (0.02, 0.05, "2-5%"),
    (0.05, 0.10, "5-10%"),
    (0.10, 0.20, "10-20%"),
    (0.20, 1.01, ">20%"),
)

MIN_SUBGROUP = 30
# Minimum per CLASS, not per group: an AUC needs both classes populated, and a floor on
# total n lets a group through on its negatives alone.
MIN_SUBGROUP_CLASS = 30
# Bootstrap draws for subgroup intervals. Lower than the eval engine's 1,000 because this
# runs per evaluator per group at report-build time; enough to separate a wide interval
# from a narrow one, which is what the panel needs to convey.
SUBGROUP_BOOT = 200
# Curves are thinned to this many points before embedding (endpoints always kept).
CURVE_POINTS = 120
# Number of equal-width probability bins for the calibration (reliability) diagram.
CALIBRATION_BINS = 10
# Decision-curve threshold grid (probability thresholds for net-benefit).
DCA_THRESHOLDS = np.linspace(0.02, 0.6, 30)

_CPU_MODELS = ("lr", "rf", "xgb")
_GPU_MODELS = ("tabpfn", "tabpfn_ft", "tabicl", "tabicl_ft")

In [ ]:
#| export
def _r(v, digits: int = 4):
    """Round a numeric value for embedding; None passes through."""
    return None if v is None else round(float(v), digits)


def _downsample(x, y, k: int = CURVE_POINTS):
    """Thin a curve to ~``k`` points, always keeping both endpoints."""
    n = len(x)
    if n <= k:
        return [_r(a) for a in x], [_r(b) for b in y]
    idx = np.unique(np.linspace(0, n - 1, k).astype(int))
    return [_r(x[i]) for i in idx], [_r(y[i]) for i in idx]


def assert_no_phi(blob: str) -> None:
    """Fail loud if the serialized report data carries anything sample-identifying.

    The upstream ``*_model_results.json`` files embed thousands of real MSK DMP sample
    ids (``oof_sample_ids``); the report must only ever ship aggregates. This is the
    hard guarantee, enforced at build time — not a convention.

    Raises:
        ValueError: if a DMP-shaped id or a sample-id field survived into the output.
    """
    hits = re.findall(r"P-\d{7}", blob)
    if hits:
        raise ValueError(
            f"PHI leak: {len(hits)} sample-id-shaped strings in report data — refusing to write"
        )
    if "sample_id" in blob or "SAMPLE_ID" in blob:
        raise ValueError("PHI leak: a sample-id field name survived into report data")


In [ ]:
#| export
def _build_cohort(labels: pd.DataFrame) -> dict:
    """Aggregate the labels table into the cohort tab (counts only, no identifiers)."""
    # PATIENT_ID is not optional: the grouped split, the leakage check and the
    # clustered interval on the primary endpoint all key on it. Missing means a
    # malformed label table -- a schema error, so say which column and why rather
    # than letting pandas raise a bare KeyError three frames down.
    missing = [c for c in ("PATIENT_ID", "label") if c not in labels.columns]
    if missing:
        raise KeyError(
            f"label table is missing required column(s) {missing}; the report cannot "
            "compute patient counts, split leakage or a clustered interval without them"
        )
    cohort = {
        "n_samples": int(len(labels)),
        "n_patients": int(labels["PATIENT_ID"].nunique()),
        "label_counts": {k: int(v) for k, v in labels["label"].value_counts().items()},
        "cancer_types": {
            k: int(v) for k, v in labels["CANCER_TYPE"].value_counts().head(14).items()
        },
        "cancer_types_other": int(
            labels["CANCER_TYPE"].value_counts().iloc[14:].sum()
            if labels["CANCER_TYPE"].nunique() > 14
            else 0
        ),
        "split_composition": {},
        "patients_in_both_splits": None,
    }
    if "split" in labels.columns:
        cohort["split_composition"] = {
            str(sp): {str(k): int(v) for k, v in grp["label"].value_counts().items()}
            for sp, grp in labels.groupby("split")
        }
        # ML-integrity check surfaced in the report: a sample-level split scatters a
        # patient's timepoints across train AND test, optimistically biasing holdout
        # metrics. Counted here, warned about client-side. See the split-leakage issue.
        # Only train/test rows count: a patient with an additional EXCLUDED sample
        # (heme / insufficient data) is not leakage — grouping over every split value
        # false-flagged 14 such patients on the v0.0.32 iris report.
        tt = labels[labels["split"].isin(["train", "test"])]
        cohort["patients_in_both_splits"] = int(
            (tt.groupby("PATIENT_ID")["split"].nunique() > 1).sum()
        )
        if cohort["patients_in_both_splits"]:
            log.warning(
                "split_patient_leakage",
                n_patients=cohort["patients_in_both_splits"],
                impact="holdout metrics optimistically biased",
            )
    return cohort


def _wilson(k: int, n: int) -> tuple[float, float]:
    """Wilson score interval — correct binomial coverage at small n and extreme p,
    where the normal approximation produces impossible bounds."""
    if n == 0:
        return (float("nan"), float("nan"))
    p_hat, z = k / n, 1.96
    denom = 1 + z * z / n
    centre = p_hat + z * z / (2 * n)
    half = z * np.sqrt(p_hat * (1 - p_hat) / n + z * z / (4 * n * n))
    return ((centre - half) / denom, (centre + half) / denom)


def _cluster_bootstrap_sens(
    p_pos: np.ndarray,
    p_tn: np.ndarray,
    pat_pos: np.ndarray,
    pat_tn: np.ndarray,
    spec: float,
    n_boot: int = 400,
    seed: int = 42,
    cluster: bool = True,
) -> tuple[float | None, float | None]:
    """Patient-clustered bootstrap interval for sensitivity at a TN-anchored specificity.

    Two things a binomial interval on sample counts gets wrong here, and this covers both:

    * **Clustering.** Samples are not independent — patients contribute several timepoints,
      so a sample-level interval is too narrow by the design effect.
    * **Threshold uncertainty.** The threshold is *estimated* from the same finite TN set,
      and it moves on resampling. Holding it fixed treats an estimate as a constant.

    Patients are resampled whole on both sides and the threshold is re-derived inside every
    resample. Returns (None, None) when the inputs cannot support an interval.
    """
    if len(p_pos) == 0 or len(p_tn) < 20:
        return (None, None)
    rng = np.random.RandomState(seed)
    # cluster=False resamples SAMPLES while still re-estimating the threshold, which is
    # what isolates the clustering contribution: the ratio of the two interval widths is
    # the design effect, and everything else the two runs share cancels out.
    if cluster:
        pos_groups = {q: np.flatnonzero(pat_pos == q) for q in np.unique(pat_pos)}
        tn_groups = {q: np.flatnonzero(pat_tn == q) for q in np.unique(pat_tn)}
    else:
        pos_groups = {i: np.array([i]) for i in range(len(p_pos))}
        tn_groups = {i: np.array([i]) for i in range(len(p_tn))}
    pos_by_patient, tn_by_patient = pos_groups, tn_groups
    pos_keys = np.array(list(pos_by_patient), dtype=object)
    tn_keys = np.array(list(tn_by_patient), dtype=object)
    out = []
    for _ in range(n_boot):
        tn_idx = np.concatenate(
            [tn_by_patient[q] for q in rng.choice(tn_keys, len(tn_keys), replace=True)]
        )
        pos_idx = np.concatenate(
            [pos_by_patient[q] for q in rng.choice(pos_keys, len(pos_keys), replace=True)]
        )
        neg_b = np.sort(p_tn[tn_idx])
        thr_b = neg_b[min(int(np.ceil(spec * len(neg_b))) - 1, len(neg_b) - 1)]
        out.append(float((p_pos[pos_idx] > thr_b).mean()))
    if len(out) < 50:
        return (None, None)
    return (float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5)))


def _anchored_operating_points(
    y_arr: np.ndarray, p_arr: np.ndarray, ids: list, lab_by_id: pd.DataFrame
) -> dict:
    """Dual-anchor operating points (#123), computed from OOF scores.

    Two anchors answer two different questions and are never blended:
      * **tumor-informed true negatives** (label ``Possible ctDNA−`` AND a paired
        IMPACT tumor AND zero confirmed variants) — the MRD question, "can we tell
        shedding from non-shedding *within cancer patients*". Thresholds are
        quantiles of thousands of samples, so 98/99% specificity is estimable.
      * **healthy donors** — the screening question. Its sens@100spec threshold is
        the MAX score over a few dozen donors; a symmetric CI for a max-statistic is
        inconsistent (the argmax donor drops out of resamples, so the threshold can
        only fall), which is why only a one-sided lower bound is reported.

    Also returns the verification-bias comparison: the same score's AUC against
    verified negatives vs all negatives vs donors only — pooled negatives include
    unpaired samples that are *unlabeled*, not verified.
    """
    from sklearn.metrics import roc_auc_score

    out: dict = {}
    if not ids or len(ids) != len(y_arr):
        return out
    meta = lab_by_id.reindex(ids)
    pos = y_arr == 1
    have = lambda c: c in meta.columns  # noqa: E731

    tn = None
    if have("label") and have("has_paired_impact") and have("n_impact_confirmed"):
        tn = (
            meta["label"].eq("Possible ctDNA−").to_numpy()
            & meta["has_paired_impact"].eq(True).to_numpy()
            & meta["n_impact_confirmed"].fillna(-1).eq(0).to_numpy()
        )
    healthy = meta["label"].eq("Healthy Normal").to_numpy() if have("label") else None

    if tn is not None and tn.sum() >= 200 and pos.sum() > 0:
        neg = np.sort(p_arr[tn])
        pts = {}
        # Patient labels for the clustered interval; without them the primary endpoint
        # can still be reported, but only as a point estimate — and it says so rather
        # than falling back to a binomial interval that would understate the spread.
        pat_all = (
            meta["PATIENT_ID"].fillna(pd.Series(ids, index=meta.index)).to_numpy()
            if have("PATIENT_ID")
            else None
        )
        for spec in (0.98, 0.99):
            thr = neg[min(int(np.ceil(spec * len(neg))) - 1, len(neg) - 1)]
            sens = float((p_arr[pos] > thr).mean())
            key = f"sens_at_{int(spec * 100)}spec"
            pts[key] = _r(sens)
            if pat_all is not None:
                lo, hi = _cluster_bootstrap_sens(
                    p_arr[pos], p_arr[tn], pat_all[pos], pat_all[tn], spec
                )
                if lo is not None and hi is not None:
                    pts[f"{key}_ci"] = [_r(lo), _r(hi)]
                    # Decompose the width rather than quoting one conflated ratio.
                    # iid = same bootstrap, samples instead of patients: the difference
                    # between the two is clustering alone (the design effect). The gap
                    # between iid and a plain binomial is threshold uncertainty — the
                    # threshold is estimated from a finite TN set and moves on resampling.
                    ilo, ihi = _cluster_bootstrap_sens(
                        p_arr[pos], p_arr[tn], pat_all[pos], pat_all[tn], spec, cluster=False
                    )
                    w_binom = np.diff(_wilson(int((p_arr[pos] > thr).sum()), int(pos.sum())))[0]
                    if ilo is not None and ihi is not None and ihi > ilo:
                        pts[f"{key}_deff"] = _r(((hi - lo) / (ihi - ilo)) ** 2, 2)
                        if w_binom > 0:
                            pts[f"{key}_threshold_inflation"] = _r(
                                ((ihi - ilo) / w_binom) ** 2, 2
                            )
            pts[f"ppv_at_{int(spec * 100)}spec"] = {
                str(pv): _r(sens * pv / (sens * pv + (1 - spec) * (1 - pv)))
                for pv in (0.05, 0.10, 0.25)
            }
        pts["n_tn"] = int(tn.sum())
        pts["n_tn_patients"] = (
            int(meta.loc[tn, "PATIENT_ID"].nunique()) if have("PATIENT_ID") else None
        )
        pts["n_pos"] = int(pos.sum())
        pts["n_pos_patients"] = (
            int(meta.loc[pos, "PATIENT_ID"].nunique()) if have("PATIENT_ID") else None
        )
        pts["ci_method"] = (
            "patient-clustered bootstrap, threshold re-estimated per resample"
            if pat_all is not None
            else "unavailable — no PATIENT_ID in the label table"
        )
        out["tn_anchor"] = pts
        out["auc_vs_verified_tn"] = _r(
            roc_auc_score(y_arr[pos | tn], p_arr[pos | tn])
            if len(set(y_arr[pos | tn])) > 1
            else np.nan
        )

    if healthy is not None and healthy.sum() > 0 and pos.sum() > 0:
        thr = p_arr[healthy].max()
        out["donor_anchor"] = {
            # One-sided: the max-statistic threshold can only fall on resampling, so
            # the observed sensitivity is a LOWER bound, not a point estimate.
            "sens_at_100spec_lower_bound": _r(float((p_arr[pos] > thr).mean())),
            "n_donors": int(healthy.sum()),
        }
        if len(set(y_arr[pos | healthy])) > 1:
            out["auc_vs_donors"] = _r(
                roc_auc_score(y_arr[pos | healthy], p_arr[pos | healthy])
            )

    if len(set(y_arr)) > 1:
        out["auc_vs_all_negatives"] = _r(roc_auc_score(y_arr, p_arr))

    # ── specificity/sensitivity trade-off on the TN anchor ──
    # Drawn rather than tabulated: the reader sees where the declared operating
    # points sit on a curve, and that the donor point is a boundary (one-sided).
    if tn is not None and tn.sum() >= 200 and pos.sum() > 0:
        neg = np.sort(p_arr[tn])
        curve = []
        for spec in np.round(np.arange(0.90, 0.9991, 0.005), 4):
            thr = neg[min(int(np.ceil(spec * len(neg))) - 1, len(neg) - 1)]
            curve.append((float(spec), _r(float((p_arr[pos] > thr).mean()))))
        out["sens_spec_curve"] = {
            "spec": [c[0] for c in curve],
            "sens": [c[1] for c in curve],
        }

        # ── detection vs tumor burden (the LOD curve) ──
        # The interpretive key to every other number on the page: sensitivity is a
        # burden-response, so a single sensitivity figure is meaningless without it.
        if "max_vaf" in meta.columns:
            thr98 = neg[min(int(np.ceil(0.98 * len(neg))) - 1, len(neg) - 1)]
            vaf = pd.to_numeric(meta["max_vaf"], errors="coerce").to_numpy()
            det = p_arr > thr98
            bins = []
            for lo, hi, lab in LOD_BINS:
                m = pos & (vaf > 0) & (vaf >= lo) & (vaf < hi)
                n = int(m.sum())
                if n < 20:
                    continue
                k = int(det[m].sum())
                bins.append(
                    {
                        "label": lab,
                        "n": n,
                        "detected": _r(k / n),
                        "ci": [_r(v) for v in _wilson(k, n)],
                    }
                )
            no_vaf = pos & ~(vaf > 0)
            lod = {"bins": bins, "operating_point": "98% spec vs verified TN"}
            if no_vaf.sum() >= 20:
                lod["no_snv_vaf"] = {
                    "n": int(no_vaf.sum()),
                    "detected": _r(float(det[no_vaf].mean())),
                }
            # LOD50: linear interpolation in log10(VAF) between the bracketing bins.
            mids = [
                (np.sqrt(max(lo, 1e-4) * min(hi, 1.0)), b["detected"])
                for (lo, hi, _), b in zip(
                    [x for x in LOD_BINS if any(b["label"] == x[2] for b in bins)], bins
                )
            ]
            below = [m for m in mids if m[1] < 0.5]
            above = [m for m in mids if m[1] >= 0.5]
            if below and above:
                x0, y0 = below[-1]
                x1, y1 = above[0]
                lx = np.log10(x0) + (0.5 - y0) * (np.log10(x1) - np.log10(x0)) / (
                    y1 - y0
                )
                lod["lod50_vaf"] = _r(float(10**lx))
                # TF ~ 2 x VAF for a clonal heterozygous variant in a diploid locus
                lod["lod50_tumor_fraction"] = _r(float(2 * 10**lx))
            out["lod"] = lod
    return out


In [ ]:
#| export
def _oof_extras(
    mr: dict, best: str, lab_by_id: pd.DataFrame, subgroup_ci: bool = False
) -> dict:
    """Curves, calibration, decision curve and subgroup AUCs from the OOF arrays.

    Sample ids are joined to labels IN MEMORY and discarded — only per-group
    aggregates (n, n_pos, auc) are returned.
    """
    from sklearn.metrics import precision_recall_curve, roc_auc_score, roc_curve

    out: dict = {}
    probs = mr.get(f"{best}_oof_probs")
    y = mr.get("oof_labels")
    if not probs or not y or len(probs) != len(y):
        return out
    y_arr = np.asarray(y, dtype=int)
    p_arr = np.asarray(probs, dtype=float)

    fpr, tpr, _ = roc_curve(y_arr, p_arr)
    out["roc"] = dict(zip(("fpr", "tpr"), _downsample(fpr, tpr)))
    prec, recl, _ = precision_recall_curve(y_arr, p_arr)
    out["pr"] = dict(zip(("recall", "precision"), _downsample(recl[::-1], prec[::-1])))
    out["prevalence"] = _r(y_arr.mean())

    # sens@100spec_healthy is thresholded on the max score among the OOF healthy
    # normals — surface how many there are so the report can flag the variance.
    slabels = mr.get("oof_sample_labels") or []
    out["n_healthy_oof"] = int(sum(1 for s in slabels if s == "Healthy Normal"))

    bins = np.clip((p_arr * CALIBRATION_BINS).astype(int), 0, CALIBRATION_BINS - 1)
    cal = []
    for b in range(CALIBRATION_BINS):
        m = bins == b
        if m.sum() >= 10:
            cal.append(
                {
                    "p_mean": _r(p_arr[m].mean()),
                    "obs": _r(y_arr[m].mean()),
                    "n": int(m.sum()),
                }
            )
    out["calibration"] = cal

    n_all = len(y_arr)
    nb, nb_all = [], []
    for thr in DCA_THRESHOLDS:
        w = thr / (1 - thr)
        pred = p_arr >= thr
        tp = int((pred & (y_arr == 1)).sum())
        fp = int((pred & (y_arr == 0)).sum())
        nb.append(_r(tp / n_all - fp / n_all * w))
        nb_all.append(_r(y_arr.mean() - (1 - y_arr.mean()) * w))
    out["dca"] = {
        "thresholds": [_r(t) for t in DCA_THRESHOLDS],
        "net_benefit": nb,
        "treat_all": nb_all,
    }

    ids = mr.get("oof_sample_ids")
    if ids and len(ids) == len(y):
        out["operating_points"] = _anchored_operating_points(
            y_arr, p_arr, list(ids), lab_by_id
        )
        sub: dict = defaultdict(dict)
        meta = lab_by_id.reindex(ids)
        pats = (
            meta["PATIENT_ID"].fillna(pd.Series(list(ids), index=meta.index)).to_numpy()
            if "PATIENT_ID" in meta.columns
            else None
        )
        for col, key in (("CANCER_TYPE", "cancer_type"), ("split", "split")):
            if col not in meta.columns:
                continue
            for grp, gidx in meta.groupby(col).groups.items():
                mask = meta.index.isin(gidx)
                yy, pp = y_arr[mask], p_arr[mask]
                # Gate on POSITIVES, not total n. A group can clear a 30-sample floor
                # on 874 samples while carrying 72 positives, and an AUC on 72 positives
                # is not comparable to one on 2,254 — that is the same denominator error
                # that made a histology odds ratio meaningless in the review.
                n_pos = int(yy.sum())
                n_neg = int(len(yy) - n_pos)
                if min(n_pos, n_neg) < MIN_SUBGROUP_CLASS or len(set(yy)) < 2:
                    continue
                entry = {
                    "n": int(len(yy)),
                    "n_pos": n_pos,
                    "auc": _r(roc_auc_score(yy, pp)),
                }
                # Intervals only for the pre-registered primary evaluator. Every other
                # evaluator's subgroup table is exploratory, and bootstrapping all 26 x 12
                # of them costs three minutes of report build for panels that carry no
                # declared claim — the same multiplicity discipline ANALYSIS_PLAN applies
                # to the endpoints, applied to what the report spends time computing.
                if subgroup_ci:
                    from kreview.eval_engine import _bootstrap_auc

                    lo, hi = _bootstrap_auc(
                        yy,
                        pp,
                        n_boot=SUBGROUP_BOOT,
                        groups=(pats[mask] if pats is not None else None),
                    )
                    if lo is not None:
                        entry["ci"] = [_r(lo), _r(hi)]
                # Share of this group's positives that are tumour-confirmed. Reported,
                # not interpreted: a group whose positives are largely the ambiguous
                # tier is a different measurement from one whose positives are verified,
                # and the reader cannot see that from an AUC alone.
                if "label" in meta.columns and n_pos:
                    tiers = meta["label"].to_numpy()[mask]
                    conf = int(np.count_nonzero(tiers == "True ctDNA+"))
                    entry["pct_true_pos"] = _r(conf / n_pos, 3)
                sub[key][str(grp)] = entry
        out["breakdowns"] = {
            k: dict(sorted(v.items(), key=lambda kv: -kv[1]["n"])[:12])
            for k, v in sub.items()
        }
    return out


In [ ]:
#| export
def _build_evaluator(row: pd.Series, outdir: Path, lab_by_id: pd.DataFrame) -> dict:
    """One evaluator record: scoreboard row + model detail + curves + ablation stability."""
    name = row["evaluator"]
    rec = {k: (_r(v) if isinstance(v, float) else v) for k, v in row.items()}

    mr: dict = {}
    for sub, fn in (
        ("cpu", f"{name}_model_results.json"),
        ("gpu", f"{name}_gpu_model_results.json"),
    ):
        p = outdir / "models" / sub / fn
        if p.exists():
            data = json.loads(p.read_text())
            mr.update(
                {k: v for k, v in data.items() if k != "oof_sample_ids"}
                if sub == "gpu"
                else data
            )
    if not mr:
        # Degrade AND surface: the scoreboard row still renders (with its status
        # column); the deep-dive sections are simply absent.
        log.warning("report_no_model_results", evaluator=name)
        return rec

    metrics = {}
    for m in [*_CPU_MODELS, *_GPU_MODELS]:
        if f"auc_{m}" not in mr:
            continue
        metrics[m] = {
            "auc": _r(mr.get(f"auc_{m}")),
            "ci": [_r(mr.get(f"auc_{m}_ci_lower")), _r(mr.get(f"auc_{m}_ci_upper"))],
            "auc_std": _r(mr.get(f"{m}_auc_std")),
            "fold_aucs": [_r(x) for x in mr.get(f"{m}_fold_aucs", [])],
            "sens95": _r(mr.get(f"{m}_sensitivity_at_95spec")),
            "sens99": _r(mr.get(f"{m}_sensitivity_at_99spec")),
            "sens100": _r(mr.get(f"{m}_sensitivity_at_100spec")),
            "sens100h": _r(mr.get(f"{m}_sensitivity_at_100spec_healthy")),
            "n_det100": mr.get(f"{m}_n_detected_at_100spec"),
            "n_pos": mr.get(f"{m}_n_total_positive"),
            "confusion": mr.get(f"{m}_confusion_matrix"),
            "holdout_auc": _r(mr.get(f"holdout_{m}_auc")),
            "holdout_sens100": _r(mr.get(f"holdout_{m}_sensitivity_at_100spec")),
        }
    rec["model_metrics"] = metrics

    best = row.get("best_model")
    rec["top_features"] = (mr.get(f"{best}_refit_features") or [])[:15]

    qc_path = outdir / "matrices" / "selected" / f"{name}_selection_qc.json"
    if qc_path.exists():
        qc = json.loads(qc_path.read_text())
        rec["selection"] = {
            "method": qc.get("method"),
            "n_input": qc.get("total_input_features"),
            "n_selected": qc.get("n_mrmr_selected") or qc.get("n_selected_union"),
            "n_variance_dropped": qc.get("n_variance_dropped"),
        }

    rec.update(
        _oof_extras(
            mr, best, lab_by_id, subgroup_ci=row.get("evaluator") == PRIMARY_EVALUATOR
        )
    )

    # Nested-CV feature-group ablation (ABLATE stage): winner group per outer fold,
    # per model — the selection-stability story. The per-sample fold_assignment array
    # in that file is deliberately never read into the output.
    bs_path = outdir / "ablation" / "merged" / f"{name}_best_subset.json"
    if bs_path.exists():
        bs = json.loads(bs_path.read_text())
        if not bs.get("passthrough"):
            fa = {}
            for m, md_ in (bs.get("per_model_per_fold_features") or {}).items():
                folds = md_.get("folds") or {}
                winners = Counter(
                    f.get("winner") for f in folds.values() if isinstance(f, dict)
                )
                scores = [
                    f.get("winner_score")
                    for f in folds.values()
                    if isinstance(f, dict) and f.get("winner_score") is not None
                ]
                fa[m] = {
                    "winners": dict(winners.most_common()),
                    "n_folds": len(folds),
                    "mean_winner_score": _r(np.mean(scores)) if scores else None,
                }
            rec["fold_ablation"] = {"groups": bs.get("groups"), "models": fa}
    return rec


In [ ]:
#| export
def _build_multimodal(mm_dir: Path) -> dict:
    """Multimodal tab: stacking/raw metrics per model + prep metadata (+ LOO ablation)."""
    prep_path = mm_dir / "prep_metadata.json"
    if not prep_path.exists():
        return {}
    prep = json.loads(prep_path.read_text())

    models: dict = {}
    for f in sorted(mm_dir.glob("stacking_*_results.json")):
        d = json.loads(f.read_text())
        mname = f.stem.replace("stacking_", "").replace("_results", "")
        entry = {}
        for scope in ("stacking", "raw"):
            auc = d.get(f"auc_{scope}_{mname}")
            if auc is None:
                continue
            entry[scope] = {
                "auc": _r(auc),
                "ci": [
                    _r(d.get(f"auc_{scope}_{mname}_ci_lower")),
                    _r(d.get(f"auc_{scope}_{mname}_ci_upper")),
                ],
                "fold_aucs": [_r(x) for x in d.get(f"{scope}_{mname}_fold_aucs", [])],
                "sens95": _r(d.get(f"{scope}_{mname}_sensitivity_at_95spec")),
                "sens100h": _r(
                    d.get(f"{scope}_{mname}_sensitivity_at_100spec_healthy")
                ),
                "vs_best_single": _r(d.get(f"{scope}_{mname}_vs_best_single")),
                "confusion": d.get(f"{scope}_{mname}_confusion_matrix"),
            }
            pr_c = d.get(f"{scope}_{mname}_pr_curve") or {}
            if isinstance(pr_c, dict) and "precision" in pr_c and "recall" in pr_c:
                rr, pp = _downsample(pr_c["recall"][::-1], pr_c["precision"][::-1])
                entry[scope]["pr"] = {"recall": rr, "precision": pp}
            fi = d.get(f"{scope}_{mname}_feature_importances")
            if isinstance(fi, dict):
                entry[scope]["top_importances"] = dict(
                    sorted(fi.items(), key=lambda kv: -abs(kv[1]))[:15]
                )
        models[mname] = entry

    ablation = None
    abl_path = mm_dir / "ablation_results.json"
    if abl_path.exists():
        abl = json.loads(abl_path.read_text())
        ablation = {
            "model": abl.get("ablation_model"),
            "baseline_auc": _r(abl.get("ablation_baseline_auc")),
            "fallback": abl.get("ablation_model_fallback"),
            # Pre-fix ablation files (< v0.0.33) carry phantom "evaluators" minted by
            # rsplit on two-part model suffixes ("X_tabicl" from the X_tabicl_ft
            # column) — drop them so old runs render clean; new runs never emit them.
            "deltas": {
                k: {
                    "delta": _r(v.get("delta")),
                    "auc_without": _r(v.get("auc_without")),
                }
                for k, v in (abl.get("ablation") or {}).items()
                if isinstance(v, dict)
                and "delta" in v
                and not k.endswith(
                    (
                        "_lr",
                        "_rf",
                        "_xgb",
                        "_tabpfn",
                        "_tabpfn_ft",
                        "_tabicl",
                        "_tabicl_ft",
                    )
                )
            },
        }

    return {
        "models": models,
        "best_single_evaluator": prep.get("best_single_evaluator"),
        "best_single_auc": _r(prep.get("best_single_auc")),
        "selection": prep.get("multimodal_selection"),
        "n_evaluators": prep.get("n_evaluators"),
        "stacking_shape": prep.get("stacking_shape"),
        "single_evaluator_aucs": {
            k: _r(v) for k, v in (prep.get("single_evaluator_aucs") or {}).items()
        },
        "ablation": ablation,
    }


In [ ]:
#| export
def _build_diagnostics(trace_path: Path | None) -> dict:
    """Run diagnostics from the Nextflow execution trace, when one is available.

    The trace is written when the WORKFLOW finishes, so an in-pipeline report render
    cannot see it — the section is optional by design and fills in when ``kreview
    report`` is re-run on a completed output directory.
    """
    diag: dict[str, Any] = {"processes": {}, "longest": [], "workflow": {}}
    if trace_path is None or not Path(trace_path).exists():
        return diag
    tr = pd.read_csv(trace_path, sep="\t")
    tr["proc"] = tr["process"].str.split(":").str[-1]
    for proc, grp in tr.groupby("proc"):
        ok = grp[grp["status"] == "COMPLETED"].sort_values("realtime", ascending=False)
        diag["processes"][proc] = {
            "n": int(len(grp)),
            "completed": int((grp["status"] == "COMPLETED").sum()),
            "failed": int((grp["status"] == "FAILED").sum()),
            "max_attempt": int(grp["attempt"].max()),
            # per-process worst case: the pipeline diagram's node panel reports these,
            # and the global "longest tasks" table cannot answer "which stage was slow".
            "slowest": (str(ok["duration"].iloc[0]) if len(ok) else None),
            "peak_rss": (str(ok["peak_rss"].iloc[0]) if len(ok) else None),
        }
    done = tr[tr["status"] == "COMPLETED"]
    diag["longest"] = [
        {"name": r["name"], "duration": r["duration"], "peak_rss": r["peak_rss"]}
        for _, r in done.sort_values("realtime", ascending=False).head(10).iterrows()
    ]
    diag["workflow"] = {
        "total_tasks": int(len(tr)),
        "failed_tasks": int((tr["status"] == "FAILED").sum()),
        "retries": int((tr["attempt"] > 1).sum()),
    }
    return diag


def _derive_findings(data: dict) -> list[dict]:
    """Rules-based automatic inferences from the run's own numbers.

    Deterministic, threshold-based, aggregates-only. Each finding is
    ``{"kind": "good"|"info"|"warn", "tab": <tab id>, "text": str}`` and states
    the numbers it is derived from, so a reader can check every claim against
    the tables. Lives in the data layer (not the template) so the rules are
    testable and PHI-guarded like everything else.
    """
    F: list[dict] = []

    def add(kind: str, tab: str, text: str, chart: str | None = None) -> None:
        # `chart` names the element whose title this sentence should become — the
        # claim then sits on its own evidence instead of in a separate wall of text
        # (charted findings render as titles and drop out of the findings list).
        entry: dict = {"kind": kind, "tab": tab, "text": text}
        if chart:
            entry["chart"] = chart
        F.append(entry)

    evs = data.get("evaluators") or []
    coh = data.get("cohort") or {}
    mm = data.get("multimodal") or {}

    # ── split integrity ──
    leaked = coh.get("patients_in_both_splits")
    if leaked == 0:
        add(
            "good",
            "overview",
            "Patient-grouped split intact: no patient has samples in both train "
            "and test, so holdout metrics are on fully unseen patients.",
        )
    elif leaked:
        add(
            "warn",
            "overview",
            f"{leaked} patients have samples in BOTH train and test — holdout "
            "metrics are optimistically biased (split-leakage).",
        )

    # ── evaluator health ──
    n_ok = sum(1 for e in evs if e.get("status") == "OK")
    if evs and n_ok == len(evs):
        add("good", "scoreboard", f"All {len(evs)} evaluators completed (status OK).")
    elif evs:
        bad = [e["evaluator"] for e in evs if e.get("status") != "OK"]
        add(
            "warn",
            "scoreboard",
            f"{len(bad)} of {len(evs)} evaluators degraded/failed: {', '.join(bad[:6])}.",
        )

    # ── best single evaluator + holdout agreement ──
    ranked = [e for e in evs if e.get("best_auc") is not None]
    ranked.sort(key=lambda e: e["best_auc"], reverse=True)
    if ranked:
        b = ranked[0]
        txt = (
            f"Strongest single signal: {b['evaluator']} "
            f"(best model {b.get('best_model')}, CV AUC {b['best_auc']:.3f}"
        )
        if b.get("holdout_auc") is not None:
            txt += f", holdout {b['holdout_auc']:.3f}"
        add("info", "scoreboard", txt + ").", chart="ov-aucbars")

        drops = [e["auc_drop"] for e in evs if e.get("auc_drop") is not None]
        if drops:
            med = float(np.median(drops))
            if abs(med) <= 0.01:
                add(
                    "good",
                    "scoreboard",
                    f"No systematic overfitting: median CV−holdout gap is "
                    f"{med:+.3f} across {len(drops)} evaluators.",
                )
            else:
                add(
                    "warn",
                    "scoreboard",
                    f"Median CV−holdout gap is {med:+.3f} across {len(drops)} "
                    "evaluators — CV numbers are optimistic; trust the holdout column.",
                )

        weak = [e["evaluator"] for e in ranked if e["best_auc"] < 0.6]
        if weak:
            add(
                "info",
                "scoreboard",
                f"{len(weak)} evaluators carry little signal (best AUC < 0.60): "
                f"{', '.join(weak[:6])}{'…' if len(weak) > 6 else ''}.",
            )

        gpu_best = sum(1 for e in ranked if str(e.get("best_model")) in _GPU_MODELS)
        add(
            "info",
            "scoreboard",
            f"GPU foundation models (TabPFN/TabICL) are the best model for "
            f"{gpu_best} of {len(ranked)} evaluators; classical models win the rest.",
        )

    # ── dual-anchor operating points (#123) ──
    ops = (data.get("primary") or {}).get("operating_points") or {}
    tn = ops.get("tn_anchor") or {}
    if tn.get("sens_at_98spec") is not None:
        pat = tn.get("n_tn_patients")
        add(
            "info",
            "scoreboard",
            f"MRD operating point (tumor-informed anchor): {tn['sens_at_98spec']:.1%} "
            f"sensitivity at 98% specificity against {tn['n_tn']:,} verified true "
            f"negatives"
            + (f" from {pat:,} patients" if pat else "")
            + f"; {tn.get('sens_at_99spec', float('nan')):.1%} at 99%. This is the "
            "clinical headline — thresholds are quantiles of thousands of samples.",
        )
        ppv = tn.get("ppv_at_98spec") or {}
        if ppv:
            add(
                "info",
                "scoreboard",
                "At that operating point, a positive call means PPV "
                + ", ".join(
                    f"{float(v):.0%} at {float(k):.0%} prevalence"
                    for k, v in sorted(ppv.items())
                )
                + ".",
            )
    don = ops.get("donor_anchor") or {}
    if don.get("sens_at_100spec_lower_bound") is not None:
        add(
            "warn",
            "scoreboard",
            f"Donor-anchored sens@100spec is a ONE-SIDED LOWER BOUND "
            f"(≥{don['sens_at_100spec_lower_bound']:.1%}, {don['n_donors']} donors): the "
            "threshold is the maximum score over those donors, so a symmetric "
            "confidence interval for it is inconsistent by construction.",
        )
    a_all, a_tn, a_don = (
        ops.get("auc_vs_all_negatives"),
        ops.get("auc_vs_verified_tn"),
        ops.get("auc_vs_donors"),
    )
    if a_all is not None and a_tn is not None:
        add(
            "info",
            "scoreboard",
            f"Verification bias: AUC {a_all:.3f} against all negatives vs "
            f"{a_tn:.3f} against verified true negatives"
            + (
                f" (and {a_don:.3f} against healthy donors — the contrast most of the "
                "literature reports, which flatters this assay)"
                if a_don
                else ""
            )
            + ". Pooled negatives include unpaired samples that are unlabeled, not verified.",
        )

    # ── healthy-anchor caveat ──
    n_healthy = (coh.get("label_counts") or {}).get("Healthy Normal")
    if n_healthy is not None and n_healthy < 100:
        add(
            "warn",
            "overview",
            f"sens@100spec-healthy thresholds on only {n_healthy} healthy donors — "
            "treat that operating point as high-variance (a single atypical donor "
            "moves it).",
        )

    # ── multimodal ──
    models = mm.get("models") or {}
    stacks = {
        m: d["stacking"]["auc"]
        for m, d in models.items()
        if d.get("stacking") and d["stacking"].get("auc") is not None
    }
    best_single_auc = mm.get("best_single_auc")
    if stacks and best_single_auc is not None:
        bm = max(stacks, key=stacks.get)
        lift = stacks[bm] - best_single_auc
        ci = (models[bm].get("stacking") or {}).get("ci") or [None, None]
        beyond = ci[0] is not None and ci[0] > best_single_auc
        add(
            "good" if lift > 0.02 else "info",
            "multimodal",
            f"Stacking helps: best meta-learner {bm} reaches AUC {stacks[bm]:.3f} "
            f"vs {best_single_auc:.3f} for the best single evaluator "
            f"({lift:+.3f}{', beyond its 95% CI' if beyond else ''}) — the feature "
            "families are partially complementary, not redundant.",
            chart="mm-auc",
        )
        spread = max(stacks.values()) - min(stacks.values())
        if spread <= 0.015:
            add(
                "info",
                "multimodal",
                f"The choice of meta-learner barely matters: all {len(stacks)} "
                f"stacking models land within {spread:.3f} AUC of each other — the "
                "signal is in the combined inputs, not the combiner.",
            )

    abl = mm.get("ablation") or {}
    deltas = {
        k: v.get("delta")
        for k, v in (abl.get("deltas") or {}).items()
        if v.get("delta") is not None
    }
    if deltas:
        pos = {k: d for k, d in deltas.items() if d > 0}
        top = sorted(deltas, key=lambda k: deltas[k], reverse=True)[:3]
        tot = sum(pos.values()) or 1.0
        share = sum(deltas[k] for k in top if deltas[k] > 0) / tot
        add(
            "info",
            "multimodal",
            f"Unique contribution concentrates in {', '.join(top)} "
            f"({share:.0%} of the total leave-one-out AUC drop across "
            f"{len(deltas)} evaluators).",
            chart="mm-abl",
        )
        redundant = sum(1 for d in deltas.values() if d <= 0.001)
        if redundant:
            add(
                "info",
                "multimodal",
                f"{redundant} of {len(deltas)} evaluators are individually "
                "redundant given the rest (LOO drop ≤ 0.001) — they overlap with "
                "stronger families rather than adding unique signal.",
            )
    if abl.get("fallback"):
        add(
            "warn",
            "multimodal",
            "The LOO ablation could not build the requested stacking model and "
            "loudly fell back — deltas are re-baselined against the fallback model.",
        )

    # ── diagnostics ──
    wf = (data.get("diagnostics") or {}).get("workflow") or {}
    if wf.get("total_tasks"):
        failed = wf.get("failed_tasks") or 0
        retries = wf.get("retries") or 0
        if failed and failed == retries:
            add(
                "good",
                "diagnostics",
                f"All {failed} task failures were transient and recovered by the "
                f"retry ladder ({wf['total_tasks']} tasks total).",
            )
        elif failed:
            add(
                "warn",
                "diagnostics",
                f"{failed} task failures with {retries} retries — check the "
                "process table for terminal failures.",
            )
    return F


In [ ]:
#| export
def build_report_data(
    outdir: str | Path,
    *,
    trace_path: str | Path | None = None,
    run_label: str = "",
) -> dict:
    """Assemble the full report data dict from a pipeline output directory.

    Aggregates-only by construction: per-sample arrays are consumed in memory and
    never emitted. Serialization must go through :func:`write_report_data` (or apply
    :func:`assert_no_phi`) so the no-identifiers guarantee is enforced.

    Args:
        outdir: The pipeline ``--outdir`` (contains ``labels/``, ``models/``,
            ``matrices/``, ``scoreboard_combined__all.parquet``, …).
        trace_path: Optional ``execution_trace.txt`` for the diagnostics tab.
        run_label: Free-text label shown in the report header.

    Raises:
        FileNotFoundError: if the scoreboard or labels are missing — the report is
            meaningless without them, so this fails loud rather than emitting an
            empty page.
    """
    outdir = Path(outdir)
    sb_path = outdir / "scoreboard_combined__all.parquet"
    labels_path = outdir / "labels" / "labels.parquet"
    for p, what in ((sb_path, "scoreboard"), (labels_path, "labels")):
        if not p.exists():
            raise FileNotFoundError(f"report_data: {what} not found at {p}")

    labels = pd.read_parquet(labels_path)
    lab_by_id = labels.set_index("SAMPLE_ID")
    sb = pd.read_parquet(sb_path).replace({np.nan: None})

    from kreview import __version__

    evaluators = [_build_evaluator(row, outdir, lab_by_id) for _, row in sb.iterrows()]
    data: dict[str, Any] = {
        "meta": {
            "title": "kreview evaluation report",
            "version": __version__,
            "run": run_label or outdir.name,
        },
        "cohort": _build_cohort(labels),
        "evaluators": evaluators,
        "multimodal": _build_multimodal(outdir / "models" / "multimodal"),
        "diagnostics": _build_diagnostics(trace_path),
        # The diagram's geometry is markup injected by render_page; only the join keys
        # and display names are data, so the blob stays PHI-checkable and layout-free.
        "pipeline": {"nodes": node_process_map(), "names": node_names(),
                     "clusters": sorted(CLUSTERS)},
    }
    # Operating points come from the PRE-REGISTERED score (multimodal stacking,
    # PRIMARY_MODEL) when the stacking matrix is available to supply sample ids —
    # that is the published-outdir case. In-pipeline the matrix is not staged, so
    # the report falls back to the primary evaluator's own score and says which
    # score produced the numbers (never silently mixing the two).
    stack_ops, ops_source = {}, None
    stack_path = outdir / "models" / "multimodal" / "stacking_matrix.parquet"
    stack_res = (
        outdir / "models" / "multimodal" / f"stacking_{PRIMARY_MODEL}_results.json"
    )
    if stack_path.exists() and stack_res.exists():
        try:
            smx = pd.read_parquet(stack_path, columns=["_sample_id", "_label"])
            probs = json.loads(stack_res.read_text()).get(
                f"stacking_{PRIMARY_MODEL}_oof_probs"
            )
            if probs and len(probs) == len(smx):
                stack_ops = _anchored_operating_points(
                    smx["_label"].to_numpy(dtype=int),
                    np.asarray(probs, dtype=float),
                    list(smx["_sample_id"]),
                    lab_by_id,
                )
                ops_source = f"multimodal stacking ({PRIMARY_MODEL}), OOF"
        except (OSError, ValueError, KeyError) as exc:
            log.warning("report_stacking_ops_unavailable", error=str(exc))

    # Pre-registered primary (ANALYSIS_PLAN.md) alongside the run's argmax, plus the
    # verification-bias triple — pooled negatives contain UNPAIRED samples that are
    # unlabeled rather than verified, so the honest number is the verified-TN one.
    primary = next(
        (e for e in evaluators if e.get("evaluator") == PRIMARY_EVALUATOR), None
    )
    ranked = [e for e in evaluators if e.get("best_auc") is not None]
    argmax = max(ranked, key=lambda e: e["best_auc"]) if ranked else None
    if stack_ops:
        ops = stack_ops
    else:
        # Fall back to the a-priori evaluator's own score; if that evaluator is not
        # in this run, fall back to the argmax — always naming the score the numbers
        # came from, never silently mixing scores or going blank.
        fallback = primary or argmax
        ops = (fallback or {}).get("operating_points") or {}
        if ops:
            which = (fallback or {}).get("evaluator")
            extra = (
                ""
                if which == PRIMARY_EVALUATOR
                else f"; a-priori {PRIMARY_EVALUATOR} absent from this run"
            )
            ops_source = f"single evaluator {which} (stacking scores not staged{extra})"
    data["primary"] = {
        "evaluator": PRIMARY_EVALUATOR,
        "model": PRIMARY_MODEL,
        "operating_points_source": ops_source,
        "declared_in": "ANALYSIS_PLAN.md",
        "auc": (primary or {}).get("best_auc"),
        "holdout_auc": (primary or {}).get("holdout_auc"),
        "argmax_evaluator": (argmax or {}).get("evaluator"),
        "argmax_auc": (argmax or {}).get("best_auc"),
        "operating_points": ops,
    }
    data["findings"] = _derive_findings(data)
    log.info(
        "report_data_built",
        n_evaluators=len(evaluators),
        n_multimodal_models=len(data["multimodal"].get("models", {})),
        has_trace=bool(data["diagnostics"]["workflow"]),
    )
    return data


def write_report_data(data: dict, out_path: str | Path) -> Path:
    """Serialize report data with the PHI guarantee enforced."""
    blob = json.dumps(data, separators=(",", ":"), allow_nan=False)
    assert_no_phi(blob)
    out_path = Path(out_path)
    out_path.write_text(blob)
    log.info("report_data_written", path=str(out_path), kb=len(blob) // 1024)
    return out_path


In [ ]:
#| export
def render_page(data: dict, out_html: str | Path) -> Path:
    """Inject report data + inlined plotly.js into the page template and write it.

    Fully self-contained output (no CDN): plotly.js comes from the installed
    ``plotly`` python package, so the page works on air-gapped HPC nodes. The PHI
    guarantee is enforced on both the data blob and the final page.
    """
    from importlib.resources import files

    import plotly.offline as _po

    blob = json.dumps(data, separators=(",", ":"), allow_nan=False)
    assert_no_phi(blob)

    template = files("kreview.templates").joinpath("report_page.html").read_text()
    page = template.replace("__PLOTLY_JS__", _po.get_plotlyjs(), 1)
    page = page.replace("__REPORT_DATA__", blob, 1)
    # Both layouts are computed here, not in the browser: the page shows or hides a
    # finished SVG, so the geometry that ships is the geometry that was verified.
    page = page.replace("__PIPELINE_METHODS__",
                        pipeline_svg("overview", chips=False) + pipeline_svg("standard", chips=False), 1)
    page = page.replace("__PIPELINE_DIAGNOSTICS__", pipeline_svg("standard"), 1)
    page = page.replace("__PIPELINE_MINIS__", mini_store(), 1)
    assert_no_phi(page)

    out_html = Path(out_html)
    out_html.parent.mkdir(parents=True, exist_ok=True)
    out_html.write_text(page)
    log.info("report_rendered", path=str(out_html), mb=round(len(page) / 1e6, 1))
    return out_html


def render_report(
    outdir: str | Path,
    out_html: str | Path,
    *,
    trace_path: str | Path | None = None,
    run_label: str = "",
) -> Path:
    """Build the report data from ``outdir`` and render the single-page report."""
    data = build_report_data(outdir, trace_path=trace_path, run_label=run_label)
    return render_page(data, out_html)